# Experiment 6 - Convolutional Neural Network for MNIST Digit Recognition

**Aim:** Build and train a convolutional neural network (CNN) that classifies
handwritten digits from the MNIST dataset.

**Theory in brief:** A CNN exploits the spatial structure of images.
Convolutional layers apply small learnable filters across the image and build
feature maps; `relu` adds nonlinearity; max-pooling downsamples the maps while
keeping the strongest responses. This gives translation tolerance with far
fewer parameters than a fully connected network. The classifier head flattens
the feature maps, applies dropout for regularization, and uses a 10-way
softmax output trained with sparse categorical cross-entropy.

**Dataset:** MNIST, downloaded at runtime from
`https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz`.
No local dataset is used.

**Expected result (original lab record):** about **99.03% test accuracy** with
a loss near **0.028** after 5 epochs.

**Runtime:** about 2-5 minutes on a Colab GPU, longer on CPU.

## 1. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

## 2. Download and prepare MNIST

The explicit NPZ URL is used so the data source is visible in the code.
Images are normalized to `[0, 1]` and expanded with a channel dimension:
`(28, 28) -> (28, 28, 1)`.

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"


def load_mnist():
    path = tf.keras.utils.get_file(
        fname="mnist.npz",
        origin=MNIST_URL,
        cache_subdir="datasets",
    )
    with np.load(path) as data:
        return (
            (np.array(data["x_train"]), np.array(data["y_train"])),
            (np.array(data["x_test"]), np.array(data["y_test"])),
        )


(X_train_raw, y_train), (X_test_raw, y_test) = load_mnist()

X_train = (X_train_raw.astype("float32") / 255.0)[..., None]
X_test = (X_test_raw.astype("float32") / 255.0)[..., None]

print(f"Training images: {X_train.shape}")
print(f"Test images    : {X_test.shape}")
print(f"Pixel range    : [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"Training labels: {y_train.shape}, test labels: {y_test.shape}")

assert X_train.shape == (60000, 28, 28, 1)
assert X_test.shape == (10000, 28, 28, 1)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for index, ax in enumerate(axes.ravel()):
    ax.imshow(X_train[index, :, :, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"label {int(y_train[index])}", fontsize=8)
    ax.axis("off")
fig.suptitle("Sample MNIST digits", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp06_mnist_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. CNN architecture

The network follows the lab sheet:

1. Conv2D: 32 filters, 3x3, `relu`, same padding
2. MaxPooling2D
3. Conv2D: 64 filters, 3x3, `relu`, same padding
4. MaxPooling2D
5. Flatten
6. Dropout 0.25
7. Dense 128, `relu`
8. Dropout 0.5
9. Dense 10, `softmax`

Dropout reduces overfitting in the dense classifier head.

In [ ]:
def build_cnn(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1), name="digit_image"),
            tf.keras.layers.Conv2D(
                32, kernel_size=3, activation="relu", padding="same", name="conv_1"
            ),
            tf.keras.layers.MaxPooling2D(name="pool_1"),
            tf.keras.layers.Conv2D(
                64, kernel_size=3, activation="relu", padding="same", name="conv_2"
            ),
            tf.keras.layers.MaxPooling2D(name="pool_2"),
            tf.keras.layers.Flatten(name="flatten"),
            tf.keras.layers.Dropout(0.25, name="dropout_1"),
            tf.keras.layers.Dense(128, activation="relu", name="dense_128"),
            tf.keras.layers.Dropout(0.5, name="dropout_2"),
            tf.keras.layers.Dense(10, activation="softmax", name="digit_probabilities"),
        ],
        name="mnist_cnn",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_cnn()
model.summary()

## 4. Training

The model is trained for 5 epochs with batch size 128. A checkpoint keeps the
best validation-accuracy weights, and early stopping restores them if
validation stops improving.

In [ ]:
checkpoint_path = OUTPUT_DIR / "exp06_best_cnn.keras"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
        verbose=1,
    ),
]

start_time = time.perf_counter()
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=2,
    callbacks=callbacks,
)
training_seconds = time.perf_counter() - start_time

print(f"Training finished in {training_seconds:.2f} seconds")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Checkpoint: {checkpoint_path.resolve()}")

## 5. Test evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
test_probabilities = model.predict(X_test, verbose=0)
test_predictions = test_probabilities.argmax(axis=1)

print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print()
print("Classification report:")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[str(digit) for digit in range(10)],
        digits=4,
        zero_division=0,
    )
)

In [ ]:
cm = confusion_matrix(y_test, test_predictions, labels=list(range(10)))

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(cm, cmap="Blues")
ax.set_title("MNIST test confusion matrix")
ax.set_xlabel("Predicted digit")
ax.set_ylabel("True digit")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticklabels(range(10))
for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        color = "white" if cm[row, column] > cm.max() / 2 else "black"
        ax.text(column, row, str(cm[row, column]), ha="center", va="center", fontsize=8, color=color)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp06_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history.history["loss"], label="Train", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Validation", linewidth=2)
axes[0].set_title("Sparse categorical cross-entropy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train", linewidth=2)
axes[1].plot(history.history["val_accuracy"], label="Validation", linewidth=2)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.90, 1.005)
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp06_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Sample predictions

In [ ]:
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(X_test), size=16, replace=False)
sample_probabilities = model.predict(X_test[sample_indices], verbose=0)
sample_predictions = sample_probabilities.argmax(axis=1)
sample_confidences = sample_probabilities.max(axis=1)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, index, prediction, confidence in zip(
    axes.ravel(), sample_indices, sample_predictions, sample_confidences
):
    ax.imshow(X_test[index, :, :, 0], cmap="gray", vmin=0, vmax=1)
    true_label = int(y_test[index])
    color = "green" if prediction == true_label else "red"
    ax.set_title(
        f"true {true_label} / pred {int(prediction)}\nconf {confidence:.3f}",
        fontsize=9,
        color=color,
    )
    ax.axis("off")
fig.suptitle("CNN predictions on random test digits", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp06_sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Misclassified examples

In [ ]:
misclassified_indices = np.flatnonzero(test_predictions != y_test)
print(f"Number of misclassified test images: {len(misclassified_indices)}")

if len(misclassified_indices) > 0:
    n_show = min(12, len(misclassified_indices))
    fig, axes = plt.subplots(3, 4, figsize=(10, 7.5))
    for ax, index in zip(axes.ravel(), misclassified_indices[:n_show]):
        ax.imshow(X_test[index, :, :, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(
            f"true {int(y_test[index])} / pred {int(test_predictions[index])}\n"
            f"conf {test_probabilities[index].max():.3f}",
            fontsize=8,
            color="red",
        )
        ax.axis("off")
    for ax in axes.ravel()[n_show:]:
        ax.axis("off")
    fig.suptitle("Examples the CNN got wrong", y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "exp06_misclassified.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No misclassified test images.")

## 9. Inference on a single digit

In [ ]:
def predict_digit(image):
    image = np.asarray(image, dtype=np.float32)
    if image.shape != (28, 28):
        raise ValueError(f"Expected a 28x28 image, got {image.shape}")
    if image.max() > 1.0:
        image = image / 255.0
    batch = image.reshape(1, 28, 28, 1)
    probabilities = model.predict(batch, verbose=0)[0]
    digit = int(np.argmax(probabilities))
    return digit, float(probabilities[digit]), probabilities


sample_index = 0
predicted_digit, confidence, all_probabilities = predict_digit(X_test_raw[sample_index])
print(f"True label : {int(y_test[sample_index])}")
print(f"Prediction : {predicted_digit} (confidence {confidence:.4f})")
print("Probabilities:", np.round(all_probabilities, 4).tolist())

## 10. Save model and metrics

In [ ]:
model_path = OUTPUT_DIR / "exp06_mnist_cnn.keras"
model.save(model_path)

metrics = pd.DataFrame(
    [
        {
            "Test loss": float(test_loss),
            "Test accuracy": float(test_accuracy),
            "Training seconds": float(training_seconds),
        }
    ]
)
metrics.to_csv(OUTPUT_DIR / "exp06_metrics.csv", index=False)
pd.DataFrame(history.history).to_csv(OUTPUT_DIR / "exp06_history.csv", index=False)

print(f"Saved final model to: {model_path.resolve()}")
print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

reloaded = tf.keras.models.load_model(model_path)
reloaded_predictions = reloaded.predict(X_test, verbose=0).argmax(axis=1)
assert np.array_equal(reloaded_predictions, test_predictions)
print("Reload check passed: predictions are identical.")

## 11. Conclusion

The two-block CNN reaches roughly **99% test accuracy** on MNIST. Convolution
and pooling exploit local spatial structure, while dropout in the classifier
head controls overfitting. The confusion matrix shows that the remaining
errors are concentrated among visually similar digits.

The original lab sheet records **99.03% accuracy** and a test loss near
**0.028** after 5 epochs. A rerun may differ slightly because of GPU
nondeterminism and the validation split.

**Artifacts created at runtime (not committed):**

- `outputs/exp06_mnist_cnn.keras`
- `outputs/exp06_best_cnn.keras`
- `outputs/exp06_metrics.csv`
- `outputs/exp06_history.csv`
- `outputs/exp06_mnist_samples.png`
- `outputs/exp06_confusion_matrix.png`
- `outputs/exp06_training_curves.png`
- `outputs/exp06_sample_predictions.png`
- `outputs/exp06_misclassified.png`

**Execution note:** the notebook was syntax-checked during repository
preparation; the full training run was not performed locally.